# ETL Pipeline

## 1. Project Configuration

In [3]:
# ==========================================
# Project Configuration
# ==========================================

from pathlib import Path

project_root = Path.cwd().parent

raw_data = project_root / "data" / "raw" / "2021-22-crdc-data.zip"
clean_data = project_root / "data" / "processed" / "crdc_bullying_clean.csv"

## 2. Load Raw Dataset

In [4]:
# Step 1 — Imports

from pathlib import Path
import zipfile
import pandas as pd

In [5]:
# Step 2 — Load the ZIP

from pathlib import Path

project_root = Path.cwd().parent

zip_path = project_root / "data" / "raw" / "2021-22-crdc-data.zip"

In [6]:
# Step 3 — Load the Harassment & Bullying dataset

with zipfile.ZipFile(zip_path, "r") as z:
    with z.open("SCH/Harassment and Bullying.csv") as f:
        df = pd.read_csv(
    f,
    dtype={
        "LEAID": str,
        "COMBOKEY": str
    },
    low_memory=False
)

df.head()

,LEA_STATE,LEA_STATE_NAME,LEAID,LEA_NAME,SCHID,SCH_NAME,COMBOKEY,JJ,SCH_HBALLEGATIONS_SEX,SCH_HBALLEGATIONS_ORI,...,SCH_HBDISCIPLINED_DIS_TR_M,SCH_HBDISCIPLINED_DIS_TR_F,TOT_HBDISCIPLINED_DIS_M,TOT_HBDISCIPLINED_DIS_F,SCH_HBDISCIPLINED_DIS_EL_M,SCH_HBDISCIPLINED_DIS_EL_F,SCH_HBDISCIPLINED_DIS_IDEA_M,SCH_HBDISCIPLINED_DIS_IDEA_F,SCH_HBDISCIPLINED_DIS_504_M,SCH_HBDISCIPLINED_DIS_504_F
0,AL,ALABAMA,0100002,Alabama Youth Services,99995,AUTAUGA CAMPUS,010000299995,Yes,0,0,...,0,0,0,0,0,0,0,0,0,0
1,AL,ALABAMA,0100005,Albertville City,870,Albertville Middle School,010000500870,No,2,0,...,0,0,0,0,0,0,0,0,0,0
2,AL,ALABAMA,0100005,Albertville City,871,Albertville High School,010000500871,No,4,0,...,0,0,0,0,0,0,0,0,0,0
3,AL,ALABAMA,0100005,Albertville City,879,Albertville Intermediate School,010000500879,No,1,0,...,0,0,0,0,0,0,0,0,0,0
4,AL,ALABAMA,0100005,Albertville City,889,Albertville Elementary School,010000500889,No,0,0,...,0,0,0,0,0,0,0,0,0,0


In [7]:
# Step 4 — Verify the dataset

print(df.shape)
df.info()

(98010, 159)
<class 'pandas.DataFrame'>
RangeIndex: 98010 entries, 0 to 98009
Columns: 159 entries, LEA_STATE to SCH_HBDISCIPLINED_DIS_504_F
dtypes: int64(152), str(7)
memory usage: 118.9 MB


In [8]:
# Step 5 — Save a backup copy

project_df = df.copy()

## 3. Rename Variables

In [9]:
# Step 7 — Define columns of interest

# Metadata columns
metadata_cols = [
    "LEA_STATE",
    "LEA_STATE_NAME",
    "LEAID",
    "LEA_NAME",
    "SCHID",
    "SCH_NAME",
    "COMBOKEY",
    "JJ"
]

# Allegation columns
allegation_cols = [col for col in df.columns if col.startswith("SCH_HBALLEGATIONS_")]

# Reported student columns
reported_cols = [col for col in df.columns if col.startswith("SCH_HBREPORTED_") or col.startswith("TOT_HBREPORTED_")]

# Disciplined student columns
disciplined_cols = [col for col in df.columns if col.startswith("SCH_HBDISCIPLINED_") or col.startswith("TOT_HBDISCIPLINED_")]

In [10]:
# step 8 — Print the number of columns in each category

print(f"Metadata:      {len(metadata_cols)}")
print(f"Allegations:  {len(allegation_cols)}")
print(f"Reported:     {len(reported_cols)}")
print(f"Disciplined:  {len(disciplined_cols)}")

Metadata:      8
Allegations:  19
Reported:     66
Disciplined:  66


In [11]:
# Step 9 — Create a new DataFrame with selected columns

selected_columns = (
    metadata_cols +
    allegation_cols +
    reported_cols +
    disciplined_cols
)

project_df = df[selected_columns]

print(project_df.shape)
project_df.head()

(98010, 159)


,LEA_STATE,LEA_STATE_NAME,LEAID,LEA_NAME,SCHID,SCH_NAME,COMBOKEY,JJ,SCH_HBALLEGATIONS_SEX,SCH_HBALLEGATIONS_ORI,...,SCH_HBDISCIPLINED_DIS_TR_M,SCH_HBDISCIPLINED_DIS_TR_F,TOT_HBDISCIPLINED_DIS_M,TOT_HBDISCIPLINED_DIS_F,SCH_HBDISCIPLINED_DIS_EL_M,SCH_HBDISCIPLINED_DIS_EL_F,SCH_HBDISCIPLINED_DIS_IDEA_M,SCH_HBDISCIPLINED_DIS_IDEA_F,SCH_HBDISCIPLINED_DIS_504_M,SCH_HBDISCIPLINED_DIS_504_F
0,AL,ALABAMA,0100002,Alabama Youth Services,99995,AUTAUGA CAMPUS,010000299995,Yes,0,0,...,0,0,0,0,0,0,0,0,0,0
1,AL,ALABAMA,0100005,Albertville City,870,Albertville Middle School,010000500870,No,2,0,...,0,0,0,0,0,0,0,0,0,0
2,AL,ALABAMA,0100005,Albertville City,871,Albertville High School,010000500871,No,4,0,...,0,0,0,0,0,0,0,0,0,0
3,AL,ALABAMA,0100005,Albertville City,879,Albertville Intermediate School,010000500879,No,1,0,...,0,0,0,0,0,0,0,0,0,0
4,AL,ALABAMA,0100005,Albertville City,889,Albertville Elementary School,010000500889,No,0,0,...,0,0,0,0,0,0,0,0,0,0


In [12]:
# =====================================================
# Step 6 – Rename columns with shorter readable names
# =====================================================
# Goal:
# Convert original CRDC column names into clean, shorter snake_case names.
# This makes the dataset easier to use in DBeaver, SQL, and Tableau.

clean_df = project_df.copy()

# 1. Manual renaming for metadata columns
metadata_rename = {
    "LEA_STATE": "state_code",
    "LEA_STATE_NAME": "state",
    "LEAID": "district_id",
    "LEA_NAME": "district",
    "SCHID": "school_id",
    "SCH_NAME": "school",
    "COMBOKEY": "school_key",
    "JJ": "juvenile_justice_school"
}

# 2. Short translation maps for repeated CRDC codes
category_map = {
    "SEX": "sex",
    "ORI": "orientation",
    "RAC": "race",
    "DIS": "disability",
    "REL": "religion",
    "GI": "gender_identity"
}

student_group_map = {
    "HI": "hispanic",
    "AM": "american_indian",
    "AS": "asian",
    "HP": "pacific_islander",
    "BL": "black",
    "WH": "white",
    "TR": "multi_race",
    "EL": "english_learner",
    "IDEA": "idea",
    "504": "section_504"
}

sex_map = {
    "M": "male",
    "F": "female"
}

# 3. Function to rename one CRDC column
def rename_crdc_column(col):
    """
    Converts original CRDC variable names into shorter readable names.

    Examples:
    SCH_HBALLEGATIONS_SEX      -> allegation_sex
    TOT_HBREPORTED_SEX_M       -> total_reported_sex_male
    SCH_HBDISCIPLINED_DIS_504_F -> disciplined_disability_section_504_female
    """

    # Metadata columns
    if col in metadata_rename:
        return metadata_rename[col]

    parts = col.split("_")

    # Allegation columns
    if col.startswith("SCH_HBALLEGATIONS_"):
        category_code = parts[-1]
        category = category_map.get(category_code, category_code.lower())
        return f"allegation_{category}"

    # Reported student columns
    if "HBREPORTED" in col:
        prefix = "total_reported" if col.startswith("TOT_") else "reported"
        remaining_parts = parts[2:]

        readable_parts = []
        for p in remaining_parts:
            if p in category_map:
                readable_parts.append(category_map[p])
            elif p in student_group_map:
                readable_parts.append(student_group_map[p])
            elif p in sex_map:
                readable_parts.append(sex_map[p])
            else:
                readable_parts.append(p.lower())

        return prefix + "_" + "_".join(readable_parts)

    # Disciplined student columns
    if "HBDISCIPLINED" in col:
        prefix = "total_disciplined" if col.startswith("TOT_") else "disciplined"
        remaining_parts = parts[2:]

        readable_parts = []
        for p in remaining_parts:
            if p in category_map:
                readable_parts.append(category_map[p])
            elif p in student_group_map:
                readable_parts.append(student_group_map[p])
            elif p in sex_map:
                readable_parts.append(sex_map[p])
            else:
                readable_parts.append(p.lower())

        return prefix + "_" + "_".join(readable_parts)

    # Fallback
    return col.lower()

# 4. Apply renaming
clean_df = clean_df.rename(columns={col: rename_crdc_column(col) for col in clean_df.columns})

# 5. Verify output
print(clean_df.shape)
clean_df.head()

(98010, 159)


,state_code,state,district_id,district,school_id,school,school_key,juvenile_justice_school,allegation_sex,allegation_orientation,...,disciplined_disability_multi_race_male,disciplined_disability_multi_race_female,total_disciplined_disability_male,total_disciplined_disability_female,disciplined_disability_english_learner_male,disciplined_disability_english_learner_female,disciplined_disability_idea_male,disciplined_disability_idea_female,disciplined_disability_section_504_male,disciplined_disability_section_504_female
0,AL,ALABAMA,0100002,Alabama Youth Services,99995,AUTAUGA CAMPUS,010000299995,Yes,0,0,...,0,0,0,0,0,0,0,0,0,0
1,AL,ALABAMA,0100005,Albertville City,870,Albertville Middle School,010000500870,No,2,0,...,0,0,0,0,0,0,0,0,0,0
2,AL,ALABAMA,0100005,Albertville City,871,Albertville High School,010000500871,No,4,0,...,0,0,0,0,0,0,0,0,0,0
3,AL,ALABAMA,0100005,Albertville City,879,Albertville Intermediate School,010000500879,No,1,0,...,0,0,0,0,0,0,0,0,0,0
4,AL,ALABAMA,0100005,Albertville City,889,Albertville Elementary School,010000500889,No,0,0,...,0,0,0,0,0,0,0,0,0,0


In [13]:
# =====================================================
# Step 7 – Quality checks
# =====================================================

print(f"Rows: {clean_df.shape[0]:,}")
print(f"Columns: {clean_df.shape[1]}")

print("\nMissing values:")
print(clean_df.isna().sum().sum())

print("\nDuplicate rows:")
print(clean_df.duplicated().sum())

print("\nColumn preview:")
print(clean_df.columns.tolist()[:30])

Rows: 98,010
Columns: 159

Missing values:
0

Duplicate rows:
0

Column preview:
['state_code', 'state', 'district_id', 'district', 'school_id', 'school', 'school_key', 'juvenile_justice_school', 'allegation_sex', 'allegation_orientation', 'allegation_race', 'allegation_disability', 'allegation_religion', 'allegation_atag', 'allegation_budd', 'allegation_cath', 'allegation_east', 'allegation_hind', 'allegation_islm', 'allegation_jwit', 'allegation_jwsh', 'allegation_morm', 'allegation_multi', 'allegation_othchrn', 'allegation_othrel', 'allegation_prot', 'allegation_sikh', 'reported_sex_hispanic_male', 'reported_sex_hispanic_female', 'reported_sex_american_indian_male']


In [14]:
# =====================================================
# Step 8 – Replace CRDC reserve codes with NULL
# =====================================================
# CRDC reserve codes are not real counts.
# We convert them to missing values so they are not accidentally counted in SQL/Tableau.

import pandas as pd

reserve_codes = [-3, -4, -5, -6, -9, -12, -13]

# Metadata columns should stay unchanged
metadata_columns = [
    "state_code",
    "state",
    "district_id",
    "district",
    "school_id",
    "school",
    "school_key",
    "juvenile_justice_school"
]

# Only replace reserve codes in numeric measurement columns
measure_columns = [col for col in clean_df.columns if col not in metadata_columns]

clean_df[measure_columns] = clean_df[measure_columns].replace(reserve_codes, pd.NA)

print("Reserve codes converted to NULL.")
print(clean_df[measure_columns].isna().sum().sum())

Reserve codes converted to NULL.
446839


In [15]:
# =====================================================
# Step 9 – Export cleaned dataset
# =====================================================

output_path = project_root / "data" / "processed" / "crdc_bullying_clean.csv"

clean_df.to_csv(output_path, index=False)

print("Dataset exported successfully!")
print(output_path)

Dataset exported successfully!
/Users/yoldaserdem/Downloads/capstone/School_Bullying_Capstone/data/processed/crdc_bullying_clean.csv


In [16]:
# =====================================================
# Step 10 – Validate exported dataset
# =====================================================

validation_df = pd.read_csv(output_path)

print(validation_df.shape)
validation_df.head()

(98010, 159)


/var/folders/r8/8pgq659n1c5db2p68_rcb9rh0000gn/T/ipykernel_86173/1402683188.py:5: DtypeWarning: Columns (0: district_id, 1: school_key) have mixed types. Specify dtype option on import or set low_memory=False.
  validation_df = pd.read_csv(output_path)


,state_code,state,district_id,district,school_id,school,school_key,juvenile_justice_school,allegation_sex,allegation_orientation,...,disciplined_disability_multi_race_male,disciplined_disability_multi_race_female,total_disciplined_disability_male,total_disciplined_disability_female,disciplined_disability_english_learner_male,disciplined_disability_english_learner_female,disciplined_disability_idea_male,disciplined_disability_idea_female,disciplined_disability_section_504_male,disciplined_disability_section_504_female
0,AL,ALABAMA,100002,Alabama Youth Services,99995,AUTAUGA CAMPUS,10000299995,Yes,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,AL,ALABAMA,100005,Albertville City,870,Albertville Middle School,10000500870,No,2.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,AL,ALABAMA,100005,Albertville City,871,Albertville High School,10000500871,No,4.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,AL,ALABAMA,100005,Albertville City,879,Albertville Intermediate School,10000500879,No,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,AL,ALABAMA,100005,Albertville City,889,Albertville Elementary School,10000500889,No,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [17]:
reserve_codes = [-3, -4, -5, -6, -9, -12, -13]

for code in reserve_codes:
    count = (validation_df == code).sum().sum()
    print(f"{code}: {count}")

-3: 0
-4: 0
-5: 0
-6: 0
-9: 0
-12: 0
-13: 0


In [18]:
validation_df.isna().sum().sum()

np.int64(446839)

In [19]:
validation_df.columns.tolist()[:20]

['state_code',
 'state',
 'district_id',
 'district',
 'school_id',
 'school',
 'school_key',
 'juvenile_justice_school',
 'allegation_sex',
 'allegation_orientation',
 'allegation_race',
 'allegation_disability',
 'allegation_religion',
 'allegation_atag',
 'allegation_budd',
 'allegation_cath',
 'allegation_east',
 'allegation_hind',
 'allegation_islm',
 'allegation_jwit']

In [20]:
validation_df.sample(5)

,state_code,state,district_id,district,school_id,school,school_key,juvenile_justice_school,allegation_sex,allegation_orientation,...,disciplined_disability_multi_race_male,disciplined_disability_multi_race_female,total_disciplined_disability_male,total_disciplined_disability_female,disciplined_disability_english_learner_male,disciplined_disability_english_learner_female,disciplined_disability_idea_male,disciplined_disability_idea_female,disciplined_disability_section_504_male,disciplined_disability_section_504_female
52228,NE,NEBRASKA,3174580,NORTHWEST PUBLIC SCHOOLS,1333,NORTHWEST HIGH SCHOOL,317458001333,No,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
28545,IL,ILLINOIS,1719260,Hinckley Big Rock CUSD 429,2212,Hinckley-Big Rock Elem Sch,171926002212,No,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
14596,CA,CALIFORNIA,641190,Vista Unified,6822,Vista Academy of Visual and Performing Arts,64119006822,No,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
76703,SD,SOUTH DAKOTA,4602070,Aberdeen School District 06-1,11,Simmons Middle School - 03,460207000011,No,2.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
87713,TX,TEXAS,4844150,VICTORIA ISD,5006,HOPKINS EL,484415005006,No,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
# test after path change 

print(raw_data.exists())
print(clean_data.exists())
print(clean_df.shape)

True
True
(98010, 159)


In [ ]:
# test after path change

for code in [-3, -4, -5, -6, -9, -12, -13]:
    print(code, (validation_df == code).sum().sum())

-3 0
-4 0
-5 0
-6 0
-9 0
-12 0
-13 0


In [ ]:
# Appendix export

# ==========================================
# Export Data Dictionary
# ==========================================

import pandas as pd

# Create a mapping between original and cleaned column names
data_dictionary = pd.DataFrame({
    "Original Variable": df.columns,
    "Clean Variable": clean_df.columns
})

# Automatically assign categories
def get_category(col):
    if col.startswith(("LEA", "SCHID", "SCH_NAME", "COMBOKEY", "JJ")):
        return "Metadata"
    elif "ALLEGATION" in col.upper():
        return "Allegation"
    elif "REPORTED" in col.upper():
        return "Reported"
    elif "DISCIPLINED" in col.upper():
        return "Disciplined"
    else:
        return "Other"

data_dictionary.insert(
    0,
    "Category",
    data_dictionary["Original Variable"].apply(get_category)
)

# Empty columns to fill later
data_dictionary["Description"] = ""
data_dictionary["Notes"] = ""

# Save to Excel
dictionary_path = project_root / "documentation" / "appendix" / "Data_Dictionary.xlsx"

data_dictionary.to_excel(dictionary_path, index=False)

print("Data Dictionary created!")
print(dictionary_path)

Polished Data Dictionary created successfully!
/Users/yoldaserdem/Downloads/capstone/School_Bullying_Capstone/documentation/appendix/Data_Dictionary.xlsx
(159, 5)


,Category,Original Variable,Clean Variable,Description,Notes
0,Metadata,LEA_STATE,state_code,Two-letter abbreviation of the U.S. state wher...,Geographic identifier.
1,Metadata,LEA_STATE_NAME,state,Full name of the U.S. state where the school i...,Geographic identifier.
2,Metadata,LEAID,district_id,Unique identifier assigned to the Local Educat...,Store as text to preserve leading zeros.
3,Metadata,LEA_NAME,district,Official name of the Local Education Agency / ...,School district / LEA name.
4,Metadata,SCHID,school_id,Unique identifier assigned to the school withi...,School identifier.
5,Metadata,SCH_NAME,school,Official name of the school.,School name.
6,Metadata,COMBOKEY,school_key,Unique identifier created by combining distric...,Primary school-level key; store as text to pre...
7,Metadata,JJ,juvenile_justice_school,Indicates whether the school is a juvenile jus...,Yes/No field.
8,Allegation,SCH_HBALLEGATIONS_SEX,allegation_sex,Number of reported harassment or bullying alle...,School-level allegation count.
9,Allegation,SCH_HBALLEGATIONS_ORI,allegation_orientation,Number of reported harassment or bullying alle...,School-level allegation count.


In [26]:
# =====================================================
# Create polished Data Dictionary / Appendix
# =====================================================

import pandas as pd
from pathlib import Path

# Create mapping table
data_dictionary = pd.DataFrame({
    "Category": data_dictionary["Category"],
    "Original Variable": data_dictionary["Original Variable"],
    "Clean Variable": data_dictionary["Clean Variable"]
})

# Human-readable labels
basis_map = {
    "sex": "sex-based",
    "orientation": "sexual-orientation-based",
    "race": "race- or ethnicity-based",
    "disability": "disability-based",
    "religion": "religion-based",
    "atag": "religion-based allegations targeting atheists or agnostics",
    "budd": "religion-based allegations targeting Buddhist students",
    "cath": "religion-based allegations targeting Catholic students",
    "east": "religion-based allegations targeting Eastern Orthodox Christian students",
    "hind": "religion-based allegations targeting Hindu students",
    "islm": "religion-based allegations targeting Muslim students",
    "jwit": "religion-based allegations targeting Jehovah's Witness students",
    "jwsh": "religion-based allegations targeting Jewish students",
    "morm": "religion-based allegations targeting Mormon / Latter-day Saints students",
    "multi": "religion-based allegations targeting multiple religious groups",
    "othchrn": "religion-based allegations targeting other Christian groups",
    "othrel": "religion-based allegations targeting other religions",
    "prot": "religion-based allegations targeting Protestant students",
    "sikh": "religion-based allegations targeting Sikh students"
}

group_map = {
    "hispanic": "Hispanic",
    "american_indian": "American Indian or Alaska Native",
    "asian": "Asian",
    "pacific_islander": "Native Hawaiian or Pacific Islander",
    "black": "Black or African American",
    "white": "White",
    "multi_race": "two or more races",
    "english_learner": "English learner",
    "idea": "IDEA disability",
    "section_504": "Section 504 disability",
    "male": "male",
    "female": "female"
}

metadata_descriptions = {
    "state_code": "Two-letter abbreviation of the U.S. state where the school is located.",
    "state": "Full name of the U.S. state where the school is located.",
    "district_id": "Unique identifier assigned to the Local Education Agency / school district.",
    "district": "Official name of the Local Education Agency / school district.",
    "school_id": "Unique identifier assigned to the school within the district.",
    "school": "Official name of the school.",
    "school_key": "Unique identifier created by combining district ID and school ID.",
    "juvenile_justice_school": "Indicates whether the school is a juvenile justice facility."
}

metadata_notes = {
    "state_code": "Geographic identifier.",
    "state": "Geographic identifier.",
    "district_id": "Store as text to preserve leading zeros.",
    "district": "School district / LEA name.",
    "school_id": "School identifier.",
    "school": "School name.",
    "school_key": "Primary school-level key; store as text to preserve leading zeros.",
    "juvenile_justice_school": "Yes/No field."
}

def pretty_basis(value):
    return basis_map.get(value, value.replace("_", " ") + "-based")

def pretty_group(parts):
    labels = [group_map.get(p, p.replace("_", " ")) for p in parts]
    return " ".join(labels)

def create_description(clean_col):
    # Metadata
    if clean_col in metadata_descriptions:
        return metadata_descriptions[clean_col]

    # Allegations
    if clean_col.startswith("allegation_"):
        basis = clean_col.replace("allegation_", "")
        label = basis_map.get(basis, basis.replace("_", " "))
        
        if basis in ["atag", "budd", "cath", "east", "hind", "islm", "jwit", "jwsh", "morm", "multi", "othchrn", "othrel", "prot", "sikh"]:
            return f"Number of reported {label}."
        else:
            return f"Number of reported harassment or bullying allegations based on {label.replace('-based', '')}."

    # Total reported
    if clean_col.startswith("total_reported_"):
        rest = clean_col.replace("total_reported_", "")
        parts = rest.split("_")
        basis = parts[0]
        group = pretty_group(parts[1:])
        return f"Total number of {group} students reported as affected by {pretty_basis(basis)} harassment or bullying."

    # Reported
    if clean_col.startswith("reported_"):
        rest = clean_col.replace("reported_", "")
        parts = rest.split("_")
        basis = parts[0]
        group = pretty_group(parts[1:])
        return f"Number of {group} students reported as affected by {pretty_basis(basis)} harassment or bullying."

    # Total disciplined
    if clean_col.startswith("total_disciplined_"):
        rest = clean_col.replace("total_disciplined_", "")
        parts = rest.split("_")
        basis = parts[0]
        group = pretty_group(parts[1:])
        return f"Total number of {group} students disciplined in relation to {pretty_basis(basis)} harassment or bullying."

    # Disciplined
    if clean_col.startswith("disciplined_"):
        rest = clean_col.replace("disciplined_", "")
        parts = rest.split("_")
        basis = parts[0]
        group = pretty_group(parts[1:])
        return f"Number of {group} students disciplined in relation to {pretty_basis(basis)} harassment or bullying."

    return "Description to be reviewed."

def create_notes(clean_col):
    if clean_col in metadata_notes:
        return metadata_notes[clean_col]

    if clean_col.startswith("allegation_"):
        return "School-level allegation count."

    if clean_col.startswith("reported_") or clean_col.startswith("total_reported_"):
        return "Student count; not an individual incident-level record."

    if clean_col.startswith("disciplined_") or clean_col.startswith("total_disciplined_"):
        return "Student count; disciplinary counts are not linked one-to-one with individual allegations."

    return ""

# Apply descriptions and notes
data_dictionary["Description"] = data_dictionary["Clean Variable"].apply(create_description)
data_dictionary["Notes"] = data_dictionary["Clean Variable"].apply(create_notes)

# Export
dictionary_path = project_root / "documentation" / "appendix" / "Data_Dictionary.xlsx"
data_dictionary.to_excel(dictionary_path, index=False)

print("Polished Data Dictionary created successfully!")
print(dictionary_path)
print(data_dictionary.shape)
data_dictionary.head(20)

Polished Data Dictionary created successfully!
/Users/yoldaserdem/Downloads/capstone/School_Bullying_Capstone/documentation/appendix/Data_Dictionary.xlsx
(159, 5)


,Category,Original Variable,Clean Variable,Description,Notes
0,Metadata,LEA_STATE,state_code,Two-letter abbreviation of the U.S. state wher...,Geographic identifier.
1,Metadata,LEA_STATE_NAME,state,Full name of the U.S. state where the school i...,Geographic identifier.
2,Metadata,LEAID,district_id,Unique identifier assigned to the Local Educat...,Store as text to preserve leading zeros.
3,Metadata,LEA_NAME,district,Official name of the Local Education Agency / ...,School district / LEA name.
4,Metadata,SCHID,school_id,Unique identifier assigned to the school withi...,School identifier.
5,Metadata,SCH_NAME,school,Official name of the school.,School name.
6,Metadata,COMBOKEY,school_key,Unique identifier created by combining distric...,Primary school-level key; store as text to pre...
7,Metadata,JJ,juvenile_justice_school,Indicates whether the school is a juvenile jus...,Yes/No field.
8,Allegation,SCH_HBALLEGATIONS_SEX,allegation_sex,Number of reported harassment or bullying alle...,School-level allegation count.
9,Allegation,SCH_HBALLEGATIONS_ORI,allegation_orientation,Number of reported harassment or bullying alle...,School-level allegation count.
